# Lab 7 — اجعل Bayan أسرع وشغّله كخدمة
**SDA-AIE-211 **

## الهدف
سنقوم بأربع خطوات فقط:
1. نقيس سرعة نموذج التصنيف.
2. نحوله إلى ONNX ثم INT8 ونقارن السرعة والنتائج.
3. نشغّل النموذج داخل API.
4. نختبر الخدمة ونوثّق النتائج.

**قبل البدء:** أكمل Lab 3a، وتأكد من وجود:
- `bayan_topic_model`
- `data/bench_mix.npy`

> شغّل الخلايا بالترتيب. لا تكتب أن الهدف تحقق إلا إذا أظهرت نتائج تشغيلك ذلك.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. تثبيت المكتبات وتحميل النموذج

In [2]:
# تثبيت المكتبات المطلوبة (يكفي تشغيله مرة واحدة، ويمكنك وضع # قبله بعد ذلك)
# optimum حُذفت لأنها تتعارض مع إصدار transformers الحديث في Colab؛ نصدّر ONNX بـ torch مباشرة
%pip -q install transformers onnx onnxscript onnxruntime fastapi uvicorn httpx torch numpy

# المكتبات: نكتب فائدة كل مكتبة بجانب اسمها
from pathlib import Path                         # التعامل مع مسارات الملفات
import time                                      # قياس زمن التنفيذ
import re                                        # تنظيف النصوص
import shutil                                    # العثور على أمر تصدير ONNX
import subprocess                                # تشغيل أمر التصدير
import threading                                 # تشغيل API في الخلفية
import numpy as np                               # حساب p50 و p99 ومقارنة النتائج
import torch                                     # تشغيل نموذج PyTorch الأصلي
import onnxruntime as ort                        # تشغيل نموذج ONNX
import httpx                                     # إرسال طلبات HTTP إلى API
import uvicorn                                   # تشغيل خادم FastAPI
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # تحميل tokenizer ونموذج التصنيف
from onnxruntime.quantization import quantize_dynamic, QuantType             # ضغط نموذج ONNX إلى INT8
from fastapi import FastAPI                      # إنشاء API
from pydantic import BaseModel, Field            # التحقق من مدخلات API

# مجلد المشروع على جهازك (غيّره إذا نقلت الملفات)
BASE_DIR = Path("/content/drive")
print('مجلد العمل | working folder:', BASE_DIR.resolve())
assert BASE_DIR.exists(), f'المجلد غير موجود: {BASE_DIR}'

# البحث عن النموذج وملف الاختبار داخل المجلد وفروعه
def find_first(name, folder_only=False):
    for p in BASE_DIR.rglob(name):
        if p.is_dir() or not folder_only:
            return p
    return None

MODEL_DIR = find_first('bayan_topic_model', folder_only=True)
BENCH_FILE = find_first('bench_mix.npy')
inside = sorted(p.name for p in BASE_DIR.iterdir())   # محتويات المجلد للمساعدة عند الخطأ

assert MODEL_DIR is not None, f'مجلد bayan_topic_model غير موجود. محتويات المجلد: {inside}'
assert (MODEL_DIR / 'config.json').exists(), f'المجلد {MODEL_DIR} لا يحتوي config.json (ليس نموذج transformers)'
assert BENCH_FILE is not None, f'ملف bench_mix.npy غير موجود. محتويات المجلد: {inside}'
DATA_DIR = BENCH_FILE.parent
print('النموذج | model :', MODEL_DIR)
print('الاختبار | bench:', BENCH_FILE)

# تحميل رسائل الاختبار
texts = np.load(BENCH_FILE, allow_pickle=True).tolist()

# تحميل النموذج الأصلي
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()
torch.set_num_threads(4)  # استخدام 4 أنوية CPU

print('عدد رسائل الاختبار:', len(texts))
print('تم تحميل النموذج ✅')

مجلد العمل | working folder: /content/drive
النموذج | model : /content/drive/.shortcut-targets-by-id/1-vfyPhMTdOHfuv5cox1F4IPardyh6B0c/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data/bayan_topic_model
الاختبار | bench: /content/drive/.shortcut-targets-by-id/1-vfyPhMTdOHfuv5cox1F4IPardyh6B0c/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data/bench_mix.npy


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

عدد رسائل الاختبار: 2000
تم تحميل النموذج ✅


## 2. قياس سرعة النموذج الأصلي

سنقارن إعدادين:
- `FP32 512`: طول 512 مع Padding ثابت.
- `FP32 128`: طول 128 مع Padding حسب طول النص.

**p50** هو الزمن الوسيط، و**p99** يوضح زمن أبطأ 1% من الطلبات.

In [3]:
def predict(text, max_length, fixed_padding):
    # تحويل النص إلى أرقام ثم تشغيل النموذج
    inputs = tokenizer(
        str(text),
        max_length=max_length,
        truncation=True,
        padding='max_length' if fixed_padding else True,
        return_tensors='pt'
    )
    with torch.inference_mode():                 # لا نحتاج حسابات التدريب
        output = model(**inputs)
    return int(output.logits.argmax(-1).item())  # إرجاع رقم الفئة المتوقعة


def benchmark(function):
    times = []
    for i in range(100):                         # نكرر القياس 100 مرة
        start = time.perf_counter()
        function(texts[i % len(texts)])
        times.append((time.perf_counter() - start) * 1000)
    return {
        'p50_ms': round(float(np.percentile(times, 50)), 2),
        'p99_ms': round(float(np.percentile(times, 99)), 2)
    }


# قياس الإعدادين
fp32_512 = benchmark(lambda x: predict(x, 512, True))
fp32_128 = benchmark(lambda x: predict(x, 128, False))

print('FP32 512:', fp32_512)
print('FP32 128:', fp32_128)

FP32 512: {'p50_ms': 606.07, 'p99_ms': 1581.92}
FP32 128: {'p50_ms': 68.52, 'p99_ms': 241.77}


## 3. تصدير النموذج إلى ONNX ثم INT8

In [5]:
# 3. تصدير النموذج إلى ONNX ثم INT8
import shutil

# التصدير على قرص Colab المحلي (السبب: لا يمكن إنشاء مجلدات في جذر /content/drive)
LOCAL_DIR = Path('/content/bayan_onnx')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
onnx_dir = LOCAL_DIR / 'topic_onnx'
onnx_dir.mkdir(exist_ok=True)

# تصدير النموذج إلى ONNX باستخدام torch (بدون Optimum)
class LogitsOnly(torch.nn.Module):               # غلاف بسيط يعيد درجات الفئات (logits) فقط
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, input_ids, attention_mask):
        return self.base(input_ids=input_ids, attention_mask=attention_mask).logits

export_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, attn_implementation='eager').eval()
dummy = tokenizer(['الخدمة لا تعمل'], max_length=128, truncation=True,
                  padding='max_length', return_tensors='pt')      # مثال واحد يحدد شكل المدخلات
onnx_file = onnx_dir / 'model.onnx'
export_args = dict(
    input_names=['input_ids', 'attention_mask'], output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'sequence'},       # يقبل أي عدد رسائل وأي طول
                  'attention_mask': {0: 'batch', 1: 'sequence'},
                  'logits': {0: 'batch'}},
    opset_version=17
)
with torch.inference_mode():
    try:
        torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
                          str(onnx_file), dynamo=False, **export_args)      # الطريقة المستقرة
    except Exception as exc:
        print('الطريقة الأولى فشلت، نجرب الثانية:', str(exc)[:200])
        torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
                          str(onnx_file), dynamo=True, **export_args)       # الطريقة الجديدة (تحتاج onnxscript)
assert onnx_file.exists(), 'لم يُنشأ ملف ONNX'

# إنشاء نسخة INT8 من نموذج ONNX
int8_file = str(LOCAL_DIR / 'topic_int8.onnx')
quantize_dynamic(str(onnx_file), int8_file, weight_type=QuantType.QInt8)  # make int8 copy

# فتح النموذجين باستخدام ONNX Runtime
onnx_model = ort.InferenceSession(str(onnx_file), providers=['CPUExecutionProvider'])
int8_model = ort.InferenceSession(int8_file, providers=['CPUExecutionProvider'])
print('تم إنشاء نسختي ONNX و INT8 ✅')

# حفظ نسخة دائمة في Drive (نسخة FP32 هي خطة الرجوع rollback)
DRIVE_ROOT = Path('/content/drive/MyDrive')
if DRIVE_ROOT.exists():
    DRIVE_OUT = DRIVE_ROOT / 'Bayan_Work' / 'models'
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(onnx_dir, DRIVE_OUT / 'topic_onnx', dirs_exist_ok=True)
    shutil.copy(int8_file, DRIVE_OUT / 'topic_int8.onnx')
    print('نُسخ إلى Drive:', DRIVE_OUT)
else:
    print('⚠️ Drive غير مركّب: الملفات محفوظة محليًا فقط في', LOCAL_DIR, '(تُحذف عند إغلاق الجلسة)')

# أحجام الملفات للتوثيق في سلّم التحسين
size_mb = lambda p: round(sum(f.stat().st_size for f in Path(p).parent.glob(Path(p).name + '*')) / 1e6, 1)
print('حجم FP32 (MB):', size_mb(onnx_file), '| حجم INT8 (MB):', size_mb(int8_file))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_380/1029017179.py:31: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:208: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscrip

تم إنشاء نسختي ONNX و INT8 ✅
نُسخ إلى Drive: /content/drive/MyDrive/Bayan_Work/models
حجم FP32 (MB): 470.8 | حجم INT8 (MB): 118.2


## 4. قياس السرعة ومقارنة التوقعات

In [6]:
def predict_onnx(text, session):
    # تجهيز النص بالطريقة نفسها ثم تشغيل نموذج ONNX
    inputs = tokenizer(str(text), max_length=128, truncation=True,
                       padding='max_length', return_tensors='np')
    # نرسل المدخلات التي يحتاجها النموذج فقط
    names = [item.name for item in session.get_inputs()]
    inputs = {name: inputs[name] for name in names}
    output = session.run(None, inputs)[0]
    return int(np.argmax(output, axis=1)[0])


# قياس سرعة ONNX و INT8
onnx_speed = benchmark(lambda x: predict_onnx(x, onnx_model))
int8_speed = benchmark(lambda x: predict_onnx(x, int8_model))
print('ONNX FP32:', onnx_speed)
print('ONNX INT8:', int8_speed)

# مقارنة نتائج INT8 بنتائج النموذج الأصلي على 100 رسالة
sample = texts[:100]
original = [predict(x, 128, False) for x in sample] #original model
compressed = [predict_onnx(x, int8_model) for x in sample] #int8 model
agreement = np.mean(np.array(original) == np.array(compressed)) #إذا كان التوقعان متطابقين نحصل على True (1) حساب نسبة التطابق
print('Prediction agreement:', round(float(agreement), 3))

ONNX FP32: {'p50_ms': 108.05, 'p99_ms': 186.36}
ONNX INT8: {'p50_ms': 85.0, 'p99_ms': 139.77}
Prediction agreement: 0.97


## 5. اختيار النسخة المناسبة

نختار INT8 إذا كانت نتائجه قريبة من النموذج الأصلي. في هذا التمرين نستخدم **97% تطابق** كحد إرشادي، وليس كمعيار عام لكل المشاريع.

نحتفظ دائمًا بنسخة ONNX FP32 كخطة رجوع (Rollback).

In [9]:
# اختيار النسخة التي ستعمل داخل الخدمة
use_int8 = agreement >= 0.97
selected_model = int8_model if use_int8 else onnx_model

print('النسخة المختارة:', 'INT8' if use_int8 else 'ONNX FP32')
print('سبب الاختيار: مقارنة السرعة مع الحفاظ على تقارب التوقعات.')

النسخة المختارة: INT8
سبب الاختيار: مقارنة السرعة مع الحفاظ على تقارب التوقعات.


## 6. تجهيز التصنيف وفحوصات البداية (Canary)

الـ Canary اختبار صغير نتأكد به أن تنظيف النص وTokenizer يعملان قبل استقبال المستخدمين.

In [11]:
# ✅ TODO 1: نفس آلية التنظيف من Lab 1 (mask_pii و normalize و preprocess)
import re
import unicodedata
import pandas as pd

# Version of the preprocessing pipeline
# v1.1 = normalize و preprocess من Lab 1 بدون تغيير + mask_pii موسّعة (لا تغيّر أي رسالة من بيانات التدريب)
PREPROCESSING_VERSION = "bayan-preproc-v1.1"


# ---------- Function to hide personal information (PII) ----------
DIGIT_MAP = str.maketrans('٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹', '01234567890123456789')   # Arabic-Indic + Persian -> Western
INVISIBLE_RE = re.compile('[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff]')    # invisible chars (written as codes)
SEP = r'[\s\-.]?'                                                               # optional space / hyphen / dot between digits

# email: characters before @ + domain + extension (2+ letters)
EMAIL_RE = re.compile(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}')
# Saudi mobile: 05 | +9665 | 009665 | 9665 (without +), any separators
PHONE_RE = re.compile(r'(?<![\d+])(?:(?:\+|00)?966' + SEP + r'0?|0)5(?:' + SEP + r'\d){8}(?!\d)')
# Saudi national ID / iqama: 10 digits starting with 1 or 2, joined or separated
NID_RE = re.compile(r'(?<!\d)[12](?:' + SEP + r'\d){9}(?!\d)')


def mask_pii(text):
    # remove invisible chars and unify digits first:
    # an invisible char inside a phone number used to stop it from being masked
    text = INVISIBLE_RE.sub('', str(text)).translate(DIGIT_MAP)
    text = EMAIL_RE.sub('[EMAIL]', text)          # email first (it may contain digits)
    text = PHONE_RE.sub('[PHONE]', text)          # phone before ID
    return NID_RE.sub('[NATIONAL_ID]', text)


# ---------- Function to normalize Arabic and English text (Lab 1, unchanged) ----------
def normalize(text):

    # Convert text to Unicode NFC format
    text = unicodedata.normalize('NFC', str(text))

    # Remove Arabic Tatweel (ـ)
    text = text.replace('ـ', '')

    # Remove Arabic diacritics (Tashkeel)
    text = re.sub(r'[\u064B-\u0652]', '', text)

    # Replace URLs and tracking links
    text = re.sub(r'https*://\S+|www\.\S+', '[URL]', text)

    # Convert Arabic-Indic digits to Western digits
    text = text.translate(str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789'))

    # Unify Alef forms: إ أ آ ٱ → ا
    text = re.sub(r'[إأآٱ]', 'ا', text)

    # Unify Yaa forms: ى → ي
    text = text.replace('ى', 'ي')

    # Unify Haa forms: ة → ه
    text = text.replace('ة', 'ه')

    # Reduce a letter repeated 3 or more times to one letter
    text = re.sub(r'([^\W\d_])\1{2,}', r'\1', text)

    # Reduce repeated ! ? ؟ to one mark
    text = re.sub(r'([!?؟])\1+', r'\1', text)

    # Replace unwanted punctuation with a space, keeping ! ? ؟ and numbers such as 10:30 and 3.5
    text = re.sub(r'[،؛]|(?<!\d)[.,:;]|[.,:;](?!\d)', ' ', text)

    # Replace multiple spaces with one space
    text = re.sub(r'\s+', ' ', text)

    # Replace words that refer to the user's email
    text = re.sub(
        r'\b[وف]?(?:ايميلي|بريدي|الايميل|البريد الالكتروني)\b',
        '[EMAIL_WORD]', text
    )

    # Replace words that refer to the user's phone number (only right before [PHONE])
    text = re.sub(
        r'\b[وف]?(?:رقم هاتفي|رقم الجوال|رقم جوالي|رقمي|جوالي|my phone number|my phone|my number)\b(?= \[PHONE\])',
        '[PHONE_WORD]', text, flags=re.IGNORECASE
    )

    # Replace words that refer to the user's national ID (only right before [NATIONAL_ID])
    text = re.sub(
        r'\b[وف]?(?:رقم هويتي|رقم الهويه|هويتي|my ID)\b(?= \[NATIONAL_ID\])',
        '[ID_WORD]', text, flags=re.IGNORECASE
    )

    # Remove spaces at the beginning and end
    return text.strip()


# ---------- Function to apply preprocessing in the correct order (Lab 1) ----------
def preprocess(text):

    # Step 1: Hide personal information FIRST
    text = mask_pii(text)

    # Step 2: Normalize the text
    text = normalize(text)

    return text


def clean_text(text):
    return preprocess(text)    # نفس تنظيف التدريب في Lab 1


def classify(text):
    clean = clean_text(text)   # تنظيف نص منظف لا يغيّره، فاستدعاؤها من /v1/analyze آمن
    pred_id = predict_onnx(clean, selected_model)
    label = model.config.id2label.get(pred_id, str(pred_id))
    return {'text': clean, 'topic': label, 'model_version': 'bayan-topic-v1',
            'preprocessing_version': PREPROCESSING_VERSION}


# ---------- اختبارات البداية (Canary): أي فشل يمنع تشغيل الخدمة ----------
TAG = {'PHONE': '[PHONE]', 'NATIONAL_ID': '[NATIONAL_ID]', 'EMAIL': '[EMAIL]'}
MUST_MASK = [
    ('0551234567', 'PHONE'), ('+966 50 123 4567', 'PHONE'), ('+966501234567', 'PHONE'),
    ('00966-55-123-4567', 'PHONE'), ('966501234567', 'PHONE'), ('٠٥٥١٢٣٤٥٦٧', 'PHONE'),
    ('۰۵۵۱۲۳۴۵۶۷', 'PHONE'), ('05 51 23 45 67', 'PHONE'), ('0551.234.567', 'PHONE'),
    ('055\u200b1234567', 'PHONE'), ('رقمي0551234567شكرا', 'PHONE'),
    ('هويتي١٠٢٣٤٥٦٧٨٩', 'NATIONAL_ID'), ('ID 1 0 2 3 4 5 6 7 8 9', 'NATIONAL_ID'), ('1023-456-789', 'NATIONAL_ID'),
    ('A.B@Mail.Example.SA', 'EMAIL'), ('lina٢٥@example.org', 'EMAIL'), ('ali+bayan605@svc.example.sa', 'EMAIL'),
]
MUST_NOT_MASK = ['Error code E-8086', 'ref 2026-11-11 SAR 1200', 'CASE18865', 'لم تصل الحافلة منذ 54 دقيقة']

leaks = [s for s, k in MUST_MASK if TAG[k] not in clean_text(s)]
overmask = [s for s in MUST_NOT_MASK if any(t in clean_text(s) for t in TAG.values())]
assert not leaks, f'PII canary: لم تُخفَ: {leaks}'
assert not overmask, f'PII canary: إخفاء زائد: {overmask}'
assert clean_text('  testـ  text  ') == 'test text'
assert clean_text('جوالي:057 200 3171 شكرا') == '[PHONE_WORD] [PHONE] شكرا'
assert clean_text(clean_text('رقمي 0551234567 والإيميل a@b.com')) == clean_text('رقمي 0551234567 والإيميل a@b.com')
assert len(tokenizer('الخدمة لا تعمل في الرياض')['input_ids']) > 2
print(f'Canary checks: GREEN ✅  ({len(MUST_MASK)} صيغة PII + {len(MUST_NOT_MASK)} بدون إخفاء زائد)')

# ---------- الاستدعاء على مجموعة الاختبار الرسمية ----------
pii_file = find_first('pii_test_set.csv')
if pii_file is not None:
    pii = pd.read_csv(pii_file)
    pii['ok'] = [TAG[r.pii_type] in clean_text(r.text) for r in pii.itertuples()]
    print('PII recall:', pii.groupby('pii_type')['ok'].mean().round(3).to_dict(),
          f"| الإجمالي {pii['ok'].sum()}/{len(pii)}")

# فحص TODO 1: هل أخفت الدالة رقم الهاتف؟
if '[PHONE]' in clean_text('اتصلوا بي على 0551234567'):
    print('TODO 1 ✅ التنظيف مطابق لـ Lab 1')
else:
    print('TODO 1 ⏳ لم تُكمل بعد: رقم الهاتف لا يزال ظاهرًا في النص')
print(classify('تم خصم المبلغ مرتين من الفاتورة'))

Canary checks: GREEN ✅  (17 صيغة PII + 4 بدون إخفاء زائد)
PII recall: {'EMAIL': 1.0, 'NATIONAL_ID': 1.0, 'PHONE': 1.0} | الإجمالي 60/60
TODO 1 ✅ التنظيف مطابق لـ Lab 1
{'text': 'تم خصم المبلغ مرتين من الفاتوره', 'topic': 'billing', 'model_version': 'bayan-topic-v1', 'preprocessing_version': 'bayan-preproc-v1.1'}


## 7. إنشاء API

Use FastAPI to build one integrated API that connects the Classification, NER, and Semantic Search components to process complaints and return their results.

## 7. إنشاء API وربط مكونات Bayan

سننشئ **واجهة واحدة** بـ **FastAPI**: تكتب شكوى، فتظهر لك 3 نتائج:
1. 📌 **Classification:** موضوع الشكوى (النموذج الذي دربته في Lab 3a).
2. 🏷️ **NER:** الأسماء والأماكن داخل الشكوى (نموذج Lab 3b) ← **هذا TODO لك**.
3. 🔎 **Semantic Search:** حالات سابقة مشابهة (فهرس FAISS من اليوم الثالث) ← **مكتوب لك كمثال**.

**الأقسام التي ستعدّلها (TODO):**
- ✍️ **TODO 1:** غيّر إعدادات الصفحة (العنوان، نص الزر، لونه) وعدد الحالات المشابهة.
- ✍️ **TODO 2:** فعّل NER بنفس طريقة البحث الدلالي (سطر واحد).
- ✍️ **TODO 3:** جرّب قيمة أخرى للباراميتر `aggregation_strategy`.

> بعد أي تعديل: شغّل خلية **الإيقاف** في آخر اللاب، ثم أعد تشغيل خلية API وخلية التشغيل.

ملاحظة مهمة – حرية تطوير الواجهة (UI Customization):

لست مقيدًا بالتصميم أو الواجهة المقدمة في هذا اللاب. يمكنك تخصيص واجهة المستخدم (UI) بالكامل حسب أفكارك، وتغيير الألوان والتنسيق وطريقة عرض النتائج، أو استخدام مكتبات وأدوات أخرى مثل Streamlit أو Gradio بدلًا من الواجهة الحالية، إذا رغبت بذلك.

كن مبدعًا! الهدف هو بناء تطبيق يعمل ويربط مكونات Bayan بشكل صحيح، وليس الالتزام بتصميم محدد.

In [12]:
# 7.1 تحميل مكونات Bayan (شغّلها كما هي)
%pip -q install sentence-transformers faiss-cpu

import json
import html
import pandas as pd
import faiss                                         # البحث السريع عن أقرب الحالات
from sentence_transformers import SentenceTransformer   # تحويل النص إلى متجه (Embedding)
from transformers import pipeline                    # أدوات جاهزة للنماذج (نستخدمها في NER)

# نبحث عن الملفات داخل مجلد المشروع (الدالة find_first موجودة في الخطوة 1)
CASES_FILE = find_first('bayan_cases.csv')            # الحالات السابقة
INDEX_FILE = find_first('bayan_cases.faiss')          # فهرس البحث من اليوم الثالث
MANIFEST_FILE = find_first('bayan_search_manifest.json')   # اسم نموذج الـ Embedding
NER_DIR = find_first('bayan_ner_model', folder_only=True)  # نموذج NER من Lab 3b

search_ready = CASES_FILE is not None
if search_ready:
    cases = pd.read_csv(CASES_FILE)
    EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    if MANIFEST_FILE is not None:                     # نستخدم النموذج نفسه الذي بنى الفهرس
        EMBED_MODEL = json.load(open(MANIFEST_FILE, encoding='utf-8'))['model']
    embedder = SentenceTransformer(EMBED_MODEL)

    if INDEX_FILE is not None:
        faiss_index = faiss.read_index(str(INDEX_FILE))
    else:
        # لا يوجد فهرس: نبنيه الآن (يستغرق بضع دقائق) ونحفظه
        print('لا يوجد bayan_cases.faiss، سنبنيه الآن...')
        case_texts = (cases['title'].fillna('').astype(str) + ' ' + cases['description'].fillna('').astype(str)
                      + ' ' + cases['resolution'].fillna('').astype(str)).str.strip().tolist()
        vectors = embedder.encode(case_texts, batch_size=64, show_progress_bar=True,
                                  normalize_embeddings=True, convert_to_numpy=True).astype('float32')
        faiss_index = faiss.IndexFlatIP(vectors.shape[1])
        faiss_index.add(vectors)
        faiss.write_index(faiss_index, str(BASE_DIR / 'bayan_cases.faiss'))

    assert faiss_index.ntotal == len(cases), 'الفهرس لا يطابق ملف الحالات: أعد بناءه من اليوم الثالث'
    print('Semantic Search ✅ عدد الحالات:', len(cases))
else:
    print('Semantic Search ⚠️ ملف bayan_cases.csv غير موجود في المجلد')

print('NER model:', NER_DIR if NER_DIR is not None else '⚠️ مجلد bayan_ner_model غير موجود (أكمل Lab 3b)')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

لا يوجد bayan_cases.faiss، سنبنيه الآن...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

RuntimeError: Error in faiss::FileIOWriter::FileIOWriter(const char*) at /project/faiss/impl/io.cpp:103: Error: 'f' failed: could not open /content/drive/bayan_cases.faiss for writing: Operation not supported

In [13]:
# إنقاذ الفهرس من الذاكرة (بدون إعادة البناء) + manifest مرقّم الإصدار
import hashlib, json
from datetime import datetime, timezone

assert 'faiss_index' in globals(), 'الفهرس غير موجود في الذاكرة: هل أُعيد تشغيل الجلسة؟'
assert faiss_index.ntotal == len(cases), f'الفهرس ناقص: {faiss_index.ntotal} من {len(cases)}'

INDEX_VERSION = 'bayan-cases-v1'                          # v1 = طريقة Day 3: title + description + resolution
INDEX_FIELDS = ['title', 'description', 'resolution']
DRIVE_ROOT = Path('/content/drive/MyDrive')
SEARCH_DIR = (DRIVE_ROOT / 'Bayan_Work' / 'search') if DRIVE_ROOT.exists() else Path('/content/bayan_search')
SEARCH_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = SEARCH_DIR / f'{INDEX_VERSION}.faiss'
MANIFEST_PATH = SEARCH_DIR / f'{INDEX_VERSION}.manifest.json'


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


faiss.write_index(faiss_index, str(INDEX_PATH))
manifest = {
    'index_version': INDEX_VERSION,
    'model': EMBED_MODEL,
    'dim': int(faiss_index.d),
    'metric': 'inner_product_on_normalized (cosine)',
    'fields': INDEX_FIELDS,
    'text_recipe': 'mask_pii only, no normalize',
    'preprocessing_version': PREPROCESSING_VERSION,
    'n_cases': int(len(cases)),
    'cases_sha256': file_sha256(CASES_FILE),
    'built_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
}
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('✅ حُفظ الفهرس:', INDEX_PATH, f'({INDEX_PATH.stat().st_size / 1e6:.1f} MB)')
print('✅ حُفظ الـ manifest:', MANIFEST_PATH)

✅ حُفظ الفهرس: /content/drive/MyDrive/Bayan_Work/search/bayan-cases-v1.faiss (30.7 MB)
✅ حُفظ الـ manifest: /content/drive/MyDrive/Bayan_Work/search/bayan-cases-v1.manifest.json


In [14]:
# التحقق من حفظ الفهرس: قراءته من الملف ومقارنته بالنسخة في الذاكرة
import json, numpy as np

# 1) الملفات موجودة وحجمها منطقي
for p in [INDEX_PATH, MANIFEST_PATH]:
    assert p.exists(), f'❌ الملف غير موجود: {p}'
    print(f'✅ موجود: {p.name}  ({p.stat().st_size / 1e6:.2f} MB)')

# 2) قراءة الفهرس من الملف
saved_index = faiss.read_index(str(INDEX_PATH))
saved_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))

# 3) المطابقة: العدد والبُعد وبصمة ملف الحالات
assert saved_index.ntotal == faiss_index.ntotal == len(cases) == saved_manifest['n_cases'], '❌ عدد الحالات لا يطابق'
assert saved_index.d == faiss_index.d == saved_manifest['dim'], '❌ بُعد المتجهات لا يطابق'
assert saved_manifest['cases_sha256'] == file_sha256(CASES_FILE), '❌ ملف الحالات تغيّر بعد البناء'
print(f'✅ {saved_index.ntotal} متجه، البُعد {saved_index.d}، وبصمة ملف الحالات مطابقة')

# 4) نفس البحث على النسختين يعطي نفس النتائج بالضبط
probe = embedder.encode(['انقطعت الكهرباء في حي النرجس', 'The bus did not arrive for 54 minutes'],
                        normalize_embeddings=True).astype('float32')
s1, p1 = faiss_index.search(probe, 10)
s2, p2 = saved_index.search(probe, 10)
assert np.array_equal(p1, p2) and np.allclose(s1, s2), '❌ نتائج الملف تختلف عن الذاكرة'
print('✅ نتائج البحث من الملف مطابقة للذاكرة')

# 5) حالة معروفة تجد نفسها في المرتبة الأولى (ترتيب المتجهات = ترتيب الحالات)
i = 123
case_text = ' '.join(str(cases.iloc[i][f]) if pd.notna(cases.iloc[i][f]) else '' for f in saved_manifest['fields']).strip()
_, pos = saved_index.search(embedder.encode([case_text], normalize_embeddings=True).astype('float32'), 1)
assert int(pos[0][0]) == i, f'❌ الحالة {i} لم تجد نفسها: وجدت {int(pos[0][0])}'
print(f'✅ الحالة {cases.iloc[i]["case_id"]} وجدت نفسها في المرتبة الأولى')

print('\n🎉 الفهرس محفوظ وسليم:', INDEX_PATH)

✅ موجود: bayan-cases-v1.faiss  (30.72 MB)
✅ موجود: bayan-cases-v1.manifest.json  (0.00 MB)
✅ 20000 متجه، البُعد 384، وبصمة ملف الحالات مطابقة
✅ نتائج البحث من الملف مطابقة للذاكرة
✅ الحالة CASE00124 وجدت نفسها في المرتبة الأولى

🎉 الفهرس محفوظ وسليم: /content/drive/MyDrive/Bayan_Work/search/bayan-cases-v1.faiss


In [15]:
from fastapi.responses import HTMLResponse           # لإرجاع صفحة ويب بدل JSON
import difflib
import numpy as np

# ✍️ TODO 1: إعدادات الصفحة والبحث
PAGE_TITLE = 'بيان | تحليل الشكاوى'
BUTTON_TEXT = 'حلّل الشكوى'
BUTTON_COLOR = '#1f4e9c'
TOP_K = 5                    # عدد الحالات المشابهة التي تظهر
CANDIDATES = 100             # المرحلة الأولى: عدد المرشحين من FAISS قبل إعادة الترتيب
MIN_SCORE = 0.45             # أقل تشابه دلالي يُقبل (اضبطيه على الاستعلامات العشرين التي بلا إجابة)
API_VERSION = 'bayan-api-v2'

app = FastAPI(title="Bayan API", version=API_VERSION)


class ComplaintRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=2000, description="Arabic or English complaint")


# ================= المكوّن الأول: Classification =================
def run_classification(clean):
    return classify(clean)          # النص منظف مسبقًا بنفس تنظيف Lab 1


# ================= المكوّن الثاني: NER =================
# ✍️ TODO 2: تحميل النموذج مرة واحدة
# ✍️ TODO 3: 'first' تعطي الكلمة كاملة وسم أول جزء منها، فلا تنقسم "بجدة" إلى كيانين
#           (قارني 'simple' و 'first' و 'max' بخلية المقارنة بعد هذه الخلية)
NER_STRATEGY = 'first'
ner_pipe = (pipeline('token-classification', model=str(NER_DIR), aggregation_strategy=NER_STRATEGY)
            if NER_DIR is not None else None)

CLITICS = ('بال', 'لل', 'وال', 'فال', 'ب', 'ل', 'و', 'ف')   # الأطول أولًا


def strip_clitic(word):
    # "بجدة" -> "جدة"، "بالنرجس" -> "النرجس": نزيل السابقة فقط إذا بقي اسم معروف
    for c in CLITICS:
        if word.startswith(c) and len(word) - len(c) >= 3:
            rest = word[len(c):]
            if c.endswith('ال'):
                rest = 'ال' + rest
            if find_district(rest) or rest in ('جدة', 'الرياض', 'الدمام'):
                return rest
    return word


def run_ner(text):
    if ner_pipe is None:
        return []
    out = []
    for e in ner_pipe(text):
        word = e['word'].strip()
        item = {'text': word, 'type': e['entity_group'], 'score': round(float(e['score']), 3)}
        if e['entity_group'] == 'LOCATION':
            item['canonical'] = strip_clitic(word)
        out.append(item)
    return out


# ================= المكوّن الثالث: Semantic Search على مرحلتين =================
DISTRICTS = {'Al Anoud': 'العنود', 'Al Badiyah': 'البادية', 'Al Faisaliyah': 'الفيصلية', 'Al Hamra': 'الحمراء',
             'Al Malqa': 'الملقا', 'Al Naeem': 'النعيم', 'Al Nakheel': 'النخيل', 'Al Olaya': 'العليا',
             'Al Rawdah': 'الروضة', 'Al Safa': 'الصفا', 'Al Salamah': 'السلامة', 'Al Shati': 'الشاطئ',
             'Al Yasmin': 'الياسمين', 'An Narjis': 'النرجس', 'As Sulaimaniyah': 'السليمانية', 'Hittin': 'حطين',
             'Obhur': 'أبحر', 'Uhud': 'أحد'}
MONTHS = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October',
          'November', 'December', 'يناير', 'فبراير', 'مارس', 'أبريل', 'مايو', 'يونيو', 'يوليو', 'أغسطس',
          'سبتمبر', 'أكتوبر', 'نوفمبر', 'ديسمبر']
OUT_OF_SCOPE = ['Tabuk', 'Najran', 'Hail', 'Abha', 'Jazan', 'تبوك', 'نجران', 'حائل', 'أبها', 'جازان']
_DIST_PATTERNS = sorted([(en, en) for en in DISTRICTS] + [('حي ' + ar, en) for en, ar in DISTRICTS.items()]
                        + [(ar, en) for en, ar in DISTRICTS.items()], key=lambda x: -len(x[0]))

def search_text(text):
    return re.sub(r'\s+', ' ', mask_pii(text)).strip()


def _has_word(pattern, text):
    return re.search(r'(?<![\w])' + re.escape(pattern) + r'(?![\w])', text, flags=re.I) is not None


def find_district(text):
    for pat, en in _DIST_PATTERNS:          # الأطول أولًا: "حي النرجس" قبل "النرجس"
        if _has_word(pat, text):
            return en
    return None


def find_date(text):
    # يوم + شهر في أي مكان من النص، مع تحمّل الأخطاء الإملائية مثل Augsut و Jaunary
    for m in re.finditer(r'(\d{1,2})\s+([A-Za-z\u0600-\u06FF]+)', text):
        hit = difflib.get_close_matches(m.group(2).title(), MONTHS, n=1, cutoff=0.75)
        if hit and 1 <= int(m.group(1)) <= 31:
            return int(m.group(1)), MONTHS.index(hit[0]) % 12
    return None


def is_out_of_scope(text):
    return any(_has_word(c, text) for c in OUT_OF_SCOPE)


# بيانات الحالات الوصفية (مرة واحدة): الحي والتاريخ لكل حالة + فهرس (حي، يوم، شهر) -> صفوف
if search_ready:
    case_meta = cases['description'].map(lambda d: (find_district(d), find_date(d)))
    assert case_meta.map(lambda m: m[0] is not None and m[1] is not None).all(), 'حالات بدون حي أو تاريخ'
    meta_index = {}
    for i, (dist, date) in enumerate(case_meta):
        meta_index.setdefault((dist, *date), []).append(i)
    case_topics = cases['topic'].values
    print(f'Meta index ✅ {len(meta_index)} مجموعة (حي، يوم، شهر)')


def run_semantic_search(raw_text, topic=None):
    if not search_ready:
        return {'results': [], 'reason': 'search_disabled'}
    q_text = search_text(raw_text)                    # إخفاء PII فقط، بدون تطبيع
    if is_out_of_scope(q_text):
        return {'results': [], 'reason': 'out_of_scope_city'}

    # المرحلة 1: FAISS + مرشحو المطابقة الوصفية
    q_vec = embedder.encode([q_text], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    sims, pos = faiss_index.search(q_vec, CANDIDATES)
    cand = {int(p): float(s) for p, s in zip(pos[0], sims[0]) if p >= 0}
    dist, date = find_district(q_text), find_date(q_text)
    meta_hits = set(meta_index.get((dist, *date), [])) if (dist and date) else set()
    for i in meta_hits - cand.keys():                 # الحالة الصحيحة قد لا تكون ضمن أعلى 100 دلاليًا
        cand[i] = float(faiss_index.reconstruct(i) @ q_vec[0])

    # المرحلة 2: إعادة الترتيب = تشابه دلالي + تطابق الحي والتاريخ + تطابق الموضوع
    ranked = []
    for i, sim in cand.items():
        c_dist, c_date = case_meta.iloc[i]
        score = sim
        score += 1.0 if i in meta_hits else (0.3 if dist and c_dist == dist else 0.0)
        score += 0.1 if topic and case_topics[i] == topic else 0.0
        ranked.append((score, sim, i))
    ranked.sort(reverse=True)

    if not ranked or ranked[0][1] < MIN_SCORE and not meta_hits:
        return {'results': [], 'reason': 'below_min_score'}
    results = []
    for score, sim, i in ranked[:TOP_K]:
        row = cases.iloc[i]
        results.append({'case_id': str(row['case_id']), 'score': round(score, 3), 'similarity': round(sim, 3),
                        'meta_match': i in meta_hits, 'topic': str(row['topic']),
                        'description': str(row['description'])[:160], 'resolution': str(row['resolution'])[:160]})
    return {'results': results, 'reason': 'ok', 'parsed': {'district': dist, 'date': date}}


# ================= المسارات =================
@app.get("/health")
def health():
    return {"status": "ok", "api_version": API_VERSION, "preprocessing_version": PREPROCESSING_VERSION,
            "model_version": "bayan-topic-v1", "precision": "int8" if use_int8 else "fp32",
            "index_version": manifest['index_version'] if search_ready else None,
            "ner": NER_STRATEGY if ner_pipe is not None else "disabled"}


@app.post("/v1/classify")
def classify_only(request: ComplaintRequest):
    # مسار خفيف لقياس شرط p99 <= 40ms (التصنيف فقط)
    return run_classification(clean_text(request.text))


@app.post("/v1/analyze")
def analyze_complaint(request: ComplaintRequest):
    clean = clean_text(request.text)              # للمصنّف: نفس تنظيف التدريب في Lab 1
    raw_masked = search_text(request.text)        # لـ NER والبحث: إخفاء PII فقط
    cls = run_classification(clean)
    search = run_semantic_search(request.text, topic=cls['topic'])
    return {
        "input_text": raw_masked,                 # نعرض النص بعد إخفاء PII فقط
        "classification": cls,
        "entities": run_ner(raw_masked),
        "similar_cases": search['results'],
        "search_info": {k: v for k, v in search.items() if k != 'results'},
    }


# ================= الصفحة =================
PAGE_HTML = """<!doctype html>
<html lang="ar" dir="rtl"><head><meta charset="utf-8"><title>__TITLE__</title>
<style>
  body { font-family: Arial, sans-serif; max-width: 760px; margin: 30px auto; padding: 0 16px; background: #f6f8f7; color: #222; }
  h1 { color: __COLOR__; }
  textarea { width: 100%; height: 110px; font-size: 16px; padding: 10px; box-sizing: border-box; }
  button { background: __COLOR__; color: #fff; border: 0; border-radius: 8px; padding: 12px 22px; font-size: 17px; cursor: pointer; margin-top: 10px; }
  .card { background: #fff; border-radius: 10px; padding: 14px 18px; margin-top: 14px; box-shadow: 0 1px 4px #0002; }
  .tag { display: inline-block; background: #e3ebf6; border-radius: 6px; padding: 3px 9px; margin: 3px; }
  .muted { color: #666; font-size: 13px; }
  .ok { color: #0a7d4f; font-weight: bold; }
</style></head><body>
<h1>__TITLE__</h1>
<textarea id="t" placeholder="اكتب الشكوى هنا بالعربية أو الإنجليزية، ويُفضّل ذكر الحي والتاريخ..."></textarea><br>
<button onclick="go()">__BUTTON__</button>
<div id="out"></div>
<script>
function esc(s) { return String(s).replace(/[&<>"]/g, function (c) { return {'&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;'}[c]; }); }
var REASONS = {out_of_scope_city: 'المدينة خارج نطاق الخدمة', below_min_score: 'لا توجد حالة مشابهة بدرجة كافية', search_disabled: 'البحث غير مفعّل'};
async function go() {
  var out = document.getElementById('out');
  out.innerHTML = '<div class="card">... جاري التحليل</div>';
  var r = await fetch('/v1/analyze', {method: 'POST', headers: {'Content-Type': 'application/json'},
                                      body: JSON.stringify({text: document.getElementById('t').value})});
  if (!r.ok) { out.innerHTML = '<div class="card">⚠️ اكتب شكوى أولًا</div>'; return; }
  var d = await r.json();
  var h = '<div class="card"><b>النص بعد إخفاء البيانات الشخصية:</b><br><span class="muted">' + esc(d.input_text) + '</span></div>';
  h += '<div class="card"><b>📌 الموضوع:</b> <span class="tag">' + esc(d.classification.topic) + '</span></div>';
  h += '<div class="card"><b>🏷️ الكيانات:</b><br>';
  h += d.entities.length ? d.entities.map(function (e) {
         return '<span class="tag">' + esc(e.text) + (e.canonical && e.canonical !== e.text ? ' (' + esc(e.canonical) + ')' : '') + ' ← ' + esc(e.type) + '</span>'; }).join('') : 'لا توجد كيانات';
  h += '</div><div class="card"><b>🔎 حالات مشابهة:</b>';
  if (d.similar_cases.length) {
    h += d.similar_cases.map(function (c) {
      return '<p>' + esc(c.case_id) + ' <span class="muted">(' + c.score + ')</span> ' + (c.meta_match ? '<span class="ok">✓ نفس الحي والتاريخ</span>' : '') +
             '<br>' + esc(c.description) + '<br><span class="muted">الحل: ' + esc(c.resolution) + '</span></p>'; }).join('');
  } else { h += '<br>' + esc(REASONS[d.search_info.reason] || 'لا توجد حالات مشابهة'); }
  out.innerHTML = h + '</div>';
}
</script></body></html>"""


@app.get("/", response_class=HTMLResponse)
def home():
    return (PAGE_HTML.replace('__TITLE__', html.escape(PAGE_TITLE))
                     .replace('__BUTTON__', html.escape(BUTTON_TEXT))
                     .replace('__COLOR__', html.escape(BUTTON_COLOR)))


print("Bayan API is ready ✅", API_VERSION)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Meta index ✅ 5832 مجموعة (حي، يوم، شهر)
Bayan API is ready ✅ bayan-api-v2


In [16]:
# تقييم البحث على bayan_queries.jsonl (130 لها إجابة + 20 بلا إجابة)
q = pd.read_json(find_first('bayan_queries.jsonl'), lines=True)
case_ids = cases['case_id'].values


def faiss_only(text, k=10):
    v = embedder.encode([search_text(text)], normalize_embeddings=True).astype('float32')
    return [case_ids[p] for p in faiss_index.search(v, k)[1][0]]


def two_stage(text, k=10):
    global TOP_K
    old, TOP_K = TOP_K, k
    try:
        topic = classify(clean_text(text))['topic']
        return [r['case_id'] for r in run_semantic_search(text, topic=topic)['results']]
    finally:
        TOP_K = old


def evaluate(fn):
    ans = q[~q['no_answer']]
    rr, hit = [], []
    for t, gold in zip(ans['query'], ans['relevant_case_id']):
        ids = fn(t)
        rank = ids.index(gold) + 1 if gold in ids else 0
        hit.append(rank > 0)
        rr.append(1 / rank if rank else 0)
    noans = q[q['no_answer']]['query']
    rejected = np.mean([len(fn(t)) == 0 for t in noans])
    xl = (ans['query_language'].values != cases.set_index('case_id').loc[ans['relevant_case_id'], 'language'].values)
    return {'Recall@10': round(np.mean(hit), 3), 'MRR@10': round(np.mean(rr), 3),
            'MRR cross-lang': round(np.mean(np.array(rr)[xl]), 3), 'no-answer rejected': round(rejected, 3)}


print('FAISS فقط   :', evaluate(faiss_only))
print('مرحلتان      :', evaluate(two_stage))

# TODO 3: مقارنة استراتيجيات التجميع في NER على جمل فيها سوابق
if NER_DIR is not None:
    probes = ['انقطعت الكهرباء بجدة منذ الصباح', 'المشكلة بالنرجس في الرياض',
              'Please escalate the case of Amal Al-Shehri in Al Malqa', 'نور الشارع مطفأ في حي الصفا']
    for strat in ['simple', 'first', 'max']:
        p = pipeline('token-classification', model=str(NER_DIR), aggregation_strategy=strat)
        print(f'\n[{strat}]')
        for s in probes:
            print('  ', s, '→', [(e['word'], e['entity_group']) for e in p(s)])

FAISS فقط   : {'Recall@10': np.float64(0.308), 'MRR@10': np.float64(0.215), 'MRR cross-lang': np.float64(0.005), 'no-answer rejected': np.float64(0.0)}
مرحلتان      : {'Recall@10': np.float64(1.0), 'MRR@10': np.float64(0.962), 'MRR cross-lang': np.float64(0.935), 'no-answer rejected': np.float64(0.9)}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[simple]
   انقطعت الكهرباء بجدة منذ الصباح → [('الكهرباء', 'SERVICE'), ('بج', 'LOCATION'), ('دة', 'LOCATION')]
   المشكلة بالنرجس في الرياض → [('بالن', 'LOCATION'), ('رج', 'LOCATION'), ('س', 'LOCATION'), ('الرياض', 'LOCATION')]
   Please escalate the case of Amal Al-Shehri in Al Malqa → [('Amal Al-Shehri', 'PERSON'), ('Al Mal', 'LOCATION'), ('qa', 'LOCATION')]
   نور الشارع مطفأ في حي الصفا → [('نور', 'PERSON'), ('الشارع', 'SERVICE'), ('حي', 'LOCATION'), ('الصف', 'LOCATION'), ('ا', 'LOCATION')]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[first]


/usr/local/lib/python3.13/dist-packages/transformers/pipelines/token_classification.py:445: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


   انقطعت الكهرباء بجدة منذ الصباح → [('الكهرباء', 'SERVICE'), ('بجدة', 'LOCATION')]
   المشكلة بالنرجس في الرياض → [('بالنرجس', 'LOCATION'), ('الرياض', 'LOCATION')]
   Please escalate the case of Amal Al-Shehri in Al Malqa → [('AmalAl-Shehri', 'PERSON'), ('AlMalqa', 'LOCATION')]
   نور الشارع مطفأ في حي الصفا → [('نور', 'PERSON'), ('الشارع', 'SERVICE'), ('حي', 'LOCATION'), ('الصفا', 'LOCATION')]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[max]
   انقطعت الكهرباء بجدة منذ الصباح → [('الكهرباء', 'SERVICE'), ('بجدة', 'LOCATION')]
   المشكلة بالنرجس في الرياض → [('بالنرجس', 'LOCATION'), ('الرياض', 'LOCATION')]
   Please escalate the case of Amal Al-Shehri in Al Malqa → [('AmalAl-Shehri', 'PERSON'), ('AlMalqa', 'LOCATION')]
   نور الشارع مطفأ في حي الصفا → [('نور', 'PERSON'), ('الشارع', 'SERVICE'), ('حي', 'LOCATION'), ('الصفا', 'LOCATION')]


### تشغيل API وتجربتها

شغّل الخادم في الخلفية، ثم افتح الرابط الذي سيظهر لك في المتصفح واكتب شكوى.

> إذا أعدت تشغيل هذه الخلية فشغّل خلية **الإيقاف** (آخر خلية) أولًا، وإلا سيكون المنفذ 8000 مشغولًا.

In [20]:
# تشغيل خادم API محليًا
server = uvicorn.Server(
    uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
)

# تشغيل الخادم في Thread منفصل حتى نستطيع إرسال طلبات إليه
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()
time.sleep(3)  # انتظار قصير حتى يبدأ الخادم

# التأكد من أن الخادم يعمل
health_response = httpx.get("http://127.0.0.1:8000/health", timeout=10)
print("Health:", health_response.json())

# تجربة إرسال شكوى إلى API
test_response = httpx.post(
    "http://127.0.0.1:8000/v1/analyze",
    json={"text": "تأخر الإصلاح في حي النرجس بالرياض وتم خصم المبلغ مرتين"},
    timeout=60
)
print("Status code:", test_response.status_code)
result = test_response.json()
print("Classification :", result["classification"]["topic"])
print("Entities       :", result["entities"])
print("Similar cases  :", len(result["similar_cases"]))

# فحص الـ TODO
print('TODO 2 ✅ NER يعمل' if result["entities"] else 'TODO 2 ⏳ الكيانات فارغة: نفّذ TODO 2 (أو لا توجد كيانات في هذا النص)')

# الواجهة
print("🌐 افتح الواجهة في المتصفح: http://127.0.0.1:8000")
try:
    from google.colab import output                    # في Colab تظهر الواجهة هنا مباشرة
    output.serve_kernel_port_as_iframe(8000, height=650)
except Exception:
    pass

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


Health: {'status': 'ok', 'api_version': 'bayan-api-v2', 'preprocessing_version': 'bayan-preproc-v1.1', 'model_version': 'bayan-topic-v1', 'precision': 'int8', 'index_version': 'bayan-cases-v1', 'ner': 'first'}
Status code: 200
Classification : outage
Entities       : [{'text': 'الإصلاح', 'type': 'SERVICE', 'score': 0.981}, {'text': 'حي', 'type': 'LOCATION', 'score': 0.925, 'canonical': 'حي'}, {'text': 'النرجس', 'type': 'LOCATION', 'score': 0.992, 'canonical': 'النرجس'}, {'text': 'بالرياض', 'type': 'LOCATION', 'score': 0.995, 'canonical': 'الرياض'}, {'text': 'خصمالمبلغ', 'type': 'SERVICE', 'score': 0.937}]
Similar cases  : 5
TODO 2 ✅ NER يعمل
🌐 افتح الواجهة في المتصفح: http://127.0.0.1:8000


<IPython.core.display.Javascript object>

### جرّب التعديل بنفسك (اختياري)

- غيّر عنوان الصفحة ولون الزر ونص الزر (TODO 1) وأعد التشغيل.
- غيّر `TOP_K` إلى 5 و`MIN_SCORE` إلى 0.5 وشاهد تغيّر الحالات المشابهة.
- جرّب `aggregation_strategy='first'` في NER (TODO 3).
- أضف مسارًا جديدًا مثل `/version` (ابحث في توثيق FastAPI: fastapi.tiangolo.com ← Tutorial ← First Steps).
- غيّر اسم المسار `/v1/analyze` (غيّره أيضًا في كود الصفحة `fetch(...)`).

**مهم:** ظهور `[]` في الكيانات يعني أن NER غير مفعّل بعد؛ نفّذ TODO 2.

In [ ]:
# إيقاف الخادم بعد الانتهاء من التجربة
server.should_exit = True
api_thread.join(timeout=10)
print("Bayan API stopped.")


Bayan API stopped.
